#### 决策树
kNN能够完成很多分类任务，但无法给出数据的内在含义，决策树的主要优势就在于数据形式非常容易理解。

#### 1. 决策树的构造

##### （1） 计算信息熵

创建trees.py，计算给定数据集的熵：
```
from math import log

def calcShannonEnt(dataSet):
    numEntries = len(dataSet) # 总数据量
    labelCounts = {}
    for featVec in dataSet:
        currentLabel = featVec[-1]
        if currentLabel not in labelCounts.keys(): # 创建数据字典，键值为最后一列数值
            labelCounts[currentLabel] = 0
        labelCounts[currentLabel] += 1 # 键值记录当前类别出现的次数
    shannonEnt = 0.0
    for key in labelCounts:
        prob = float(labelCounts[key])/numEntries
        shannonEnt -= prob * log(prob,2)
    return shannonEnt
```

In [16]:
import trees
import importlib
importlib.reload(trees)

<module 'trees' from '/home/xkzhai/MLiA/trees.py'>

In [17]:
myData, labels = trees.createDataSet()
myData

[[1, 1, 'yes'], [1, 1, 'yes'], [1, 0, 'no'], [0, 1, 'no'], [0, 1, 'no']]

In [19]:
trees.calcShannonEnt(myData)

0.9709505944546686

In [20]:
myData[0][-1] = 'maybe'
myData

[[1, 1, 'maybe'], [1, 1, 'yes'], [1, 0, 'no'], [0, 1, 'no'], [0, 1, 'no']]

In [21]:
trees.calcShannonEnt(myData)

1.3709505944546687

##### （2） 划分数据集 

在trees.py中添加如下代码，按照给定特征划分数据集：
```
def splitDataSet(dataSet,axis,value):
    retDataSet = []
    for featVec in dataSet:
        if featVec[axis] == value:
            reducedFeatVec = featVec[:axis]
            reducedFeatVec.extend(featVec[axis+1:])
            retDataSet.append(reducedFeatVec)
    return retDataSet
```

##### 注：extend与append对比

In [23]:
a = [1,2,3];
b = [4,5,6];
a.append(b)
a

[1, 2, 3, [4, 5, 6]]

In [25]:
a = [1,2,3];
b = [4,5,6];
a.extend(b)
a

[1, 2, 3, 4, 5, 6]

使用前面的简单数据测试splitDataSet():

In [27]:
import importlib
importlib.reload(trees)

myData, labels = trees.createDataSet();
myData

[[1, 1, 'yes'], [1, 1, 'yes'], [1, 0, 'no'], [0, 1, 'no'], [0, 1, 'no']]

In [28]:
trees.splitDataSet(myData,0,1)

[[1, 'yes'], [1, 'yes'], [0, 'no']]

在trees.py中添加如下代码，遍历整个数据集，循环计算香农熵和splitDataSet()，找到最好的特征划分方式
```
def chooseBestFeatureToSplit(dataSet):
    numFeatures = len(dataSet[0]) - 1 # 计算特征数量
    baseEntropy = calcShannonEnt(dataSet) # 计算初始熵
    bestInfoGain = 0.0; bestFeature = -1
    for i in range(numFeatures):
        featList  = [example[i] for example in dataSet] # 取出数据集中第i个特征所有值
        uniqueVals = set(featList) # 去除重复值
        newEntropy = 0.0
        for value in uniqueVals:   # 依次按照不同的值进行抽取
            subDataSet = splitDataSet(dataSet,i,value)  
            prob = len(subDataSet) / float(len(dataSet))
            newEntropy += prob * calcShannonEnt(subDataSet)
        infoGain = baseEntropy - newEntropy
        if (infoGain > bestInfoGain):
            bestInfoGain = infoGain
            bestFeature = i
    return bestFeature
```

In [31]:
import importlib
importlib.reload(trees)

myData, labels = trees.createDataSet();
myData


[[1, 1, 'yes'], [1, 1, 'yes'], [1, 0, 'no'], [0, 1, 'no'], [0, 1, 'no']]

In [32]:
trees.chooseBestFeatureToSplit(myData)

0

In [33]:
myData

[[1, 1, 'yes'], [1, 1, 'yes'], [1, 0, 'no'], [0, 1, 'no'], [0, 1, 'no']]